# GPT-J-6B ROME Reproduction - One-Click Google Colab Notebook (v6)

Reproduce ROME (Rank-One Model Editing) on GPT-J-6B (fp16) using EasyEdit (CISC7021 Group26 W2).

**v3 change**: cell sources stored as single strings with explicit \n (no line-joining when pasted/rendered).
**v2 fixes kept**: one-shot dependency install (iopath / fairscale / av / qwen_vl_utils / hydra-core etc.), correct hydra package name (hydra-core), model download keeps `*.bin`, run output captured to log, result summary matches script fields.

**How to run**: Runtime -> Run all. Idempotent: existing clone / model / data are skipped on re-run.

In [1]:
# 1.1 Clone the project repository (robust: retry git clone, fallback to zip download)
import os, subprocess, sys, time, zipfile, urllib.request

REPO_URL = "https://github.com/FongChinWai-Vivi/CALM-Editor-CISC7021-Task1-Group26.git"
ZIP_URL = "https://github.com/FongChinWai-Vivi/CALM-Editor-CISC7021-Task1-Group26/archive/refs/heads/main.zip"
PROJ_DIR = "/content/CALM-Editor-CISC7021-Task1-Group26"
ZIP_PATH = "/content/repo.zip"
EXTRACT_DIR = "/content"
EXTRACTED_NAME = "CALM-Editor-CISC7021-Task1-Group26-main"

def clone_with_retry():
    for attempt in range(1, 4):
        try:
            print(f"Cloning project repository (attempt {attempt}/3) ...")
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJ_DIR], check=True)
            return True
        except subprocess.CalledProcessError as e:
            print(f"git clone failed (attempt {attempt}/3): {e}")
            if attempt < 3:
                time.sleep(5)
    return False

def clone_via_zip():
    print("git clone failed after 3 attempts, falling back to zip download ...")
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    extracted = os.path.join(EXTRACT_DIR, EXTRACTED_NAME)
    if os.path.exists(PROJ_DIR):
        print("Target directory already exists, keeping it.")
    else:
        os.rename(extracted, PROJ_DIR)
    print("Repository extracted from zip archive.")

if not os.path.isdir(PROJ_DIR):
    if not clone_with_retry():
        clone_via_zip()
else:
    print("Project directory already exists, skipping clone.")
print("Project directory:", PROJ_DIR)

Cloning project repository (attempt 1/3) ...
Project directory: /content/CALM-Editor-CISC7021-Task1-Group26


In [2]:
# 1.2 Check that the W2 reproduction script and GPT-J hparams are in place
script_path = os.path.join(PROJ_DIR, "EasyEdit", "examples", "w2_rome_repro.py")
hparams_path = os.path.join(PROJ_DIR, "EasyEdit", "hparams", "ROME", "gpt-j-6B.yaml")

if os.path.isfile(script_path) and os.path.isfile(hparams_path):
    print("w2_rome_repro.py and gpt-j-6B.yaml both exist, ready to run.")
else:
    print("The repository is missing the following files, please upload them:")
    print("  -", script_path)
    print("  -", hparams_path)
    from google.colab import files
    up = files.upload()
    import shutil
    for name in up.keys():
        if name.endswith("w2_rome_repro.py"):
            os.makedirs(os.path.dirname(script_path), exist_ok=True)
            shutil.move(name, script_path)
        elif name.endswith("gpt-j-6B.yaml"):
            os.makedirs(os.path.dirname(hparams_path), exist_ok=True)
            shutil.move(name, hparams_path)
    print("Upload complete.")

w2_rome_repro.py and gpt-j-6B.yaml both exist, ready to run.


In [3]:
# 1.3 Check torch version (skip reinstall by default, keep CUDA check)
import torch

print("Current torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SKIP_TORCH_REINSTALL = True
MIN_TORCH = "2.9.0"
EXPECT_TORCH = "2.9.1"

def version_tuple(v):
    nums = []
    for x in v.split(".")[:3]:
        try:
            nums.append(int(x))
        except ValueError:
            break
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums)

cur = torch.__version__.split("+")[0]
if SKIP_TORCH_REINSTALL:
    if version_tuple(cur) >= version_tuple(MIN_TORCH):
        print(f"torch {cur} is compatible, skipping reinstall.")
    else:
        print(f"torch {cur} is older than {MIN_TORCH}; set SKIP_TORCH_REINSTALL=False to reinstall torch=={EXPECT_TORCH}.")
else:
    if cur != EXPECT_TORCH:
        print(f"Current torch {cur} != target {EXPECT_TORCH}, reinstalling cu126 build ...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], check=False)
        subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.9.1", "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
        print("torch reinstalled as 2.9.1 (cu126). Please Runtime -> Restart session, then Run All again.")
        sys.exit(0)
    else:
        print("torch version satisfies the requirement, skipping reinstall.")

Current torch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
torch 2.11.0 is compatible, skipping reinstall.


In [4]:
# 1.4 Install ALL EasyEdit dependencies (one-shot, fixes the import chain)
# EasyEdit's easyeditor/__init__.py -> dataset -> trainer.utils -> blip2_models
# pulls in BLIP2 deps: iopath, fairscale, av, qwen_vl_utils, hydra-core, ...
import importlib, os, re, subprocess, sys

req_path = os.path.join(PROJ_DIR, "EasyEdit", "requirements.txt")
filtered_req = os.path.join(PROJ_DIR, "EasyEdit", "requirements_no_torch.txt")
with open(req_path, "r", encoding="utf-8") as f:
    lines = f.readlines()
kept = [ln for ln in lines if not re.match(r"^\s*(torch|torchvision|torchaudio)([=@<>\s]|$)", ln, re.IGNORECASE)]
with open(filtered_req, "w", encoding="utf-8") as f:
    f.writelines(kept)

print("Installing requirements (torch-related lines excluded) ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", filtered_req], check=False)

# Extra packages needed by the BLIP2 import chain (not in requirements.txt)
EXTRA = ["iopath", "fairscale", "av", "qwen_vl_utils", "hydra-core", "timm", "accelerate", "peft", "higher"]  # higher is required by MEND.py (import chain)
print("Installing extra deps:", EXTRA)
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + EXTRA, check=False)

# Verify the import chain works; auto-fill anything still missing (max 5 rounds)
os.chdir(os.path.join(PROJ_DIR, "EasyEdit"))
sys.path.insert(0, os.getcwd())
PKG_ALIAS = {"hydra": "hydra-core"}  # easyeditor imports hydra; the pip package is hydra-core

for attempt in range(5):
    try:
        import easyeditor
        print("easyeditor import OK - all dependencies satisfied.")
        break
    except ModuleNotFoundError as e:
        name = str(e).split("'")[1].split(".")[0]
        install = PKG_ALIAS.get(name, name)
        print(f"missing {name} -> pip install {install}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", install], check=False)
else:
    print("Still missing packages after 5 rounds.")
    raise SystemExit(1)

Installing requirements (torch-related lines excluded) ...
Installing extra deps: ['iopath', 'fairscale', 'av', 'qwen_vl_utils', 'hydra-core', 'timm', 'accelerate', 'peft', 'higher']


/content/CALM-Editor-CISC7021-Task1-Group26/EasyEdit/easyeditor/trainer/algs/higher_utils/utils.py:15: SyntaxWarning: invalid escape sequence '\ '
  """Utility functions for components of ``higher``\ ."""
/usr/local/lib/python3.13/dist-packages/timm/models/hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/CALM-Editor-CISC7021-Task1-Group26/EasyEdit/easyeditor/models/melo/peft_egg/src/peft/tuners/lora.py:233: SyntaxWarning: invalid escape sequence '\.'
  layer_index = re.match(f".*.{pattern}\.(\d+)\.*", key)
/content/CALM-Editor-CISC7021-Task1-Group26/EasyEdit/easyeditor/models/melo/peft_egg/src/peft/tuners/melo.py:183: SyntaxWarning: invalid escape sequence '\.'
  layer_index = re.match(f".*.{pattern}\.(\d+)\.*", key)


missing zhipuai -> pip install zhipuai
missing rouge -> pip install rouge
easyeditor import OK - all dependencies satisfied.


In [5]:
# 2.1 Check disk space
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   43G   70G  38% /


In [6]:
# 2.2 Download the model (idempotent; keeps *.bin and detects pytorch_model.bin)
from huggingface_hub import snapshot_download

model_dir = os.path.join(PROJ_DIR, "EasyEdit", "hugging_cache", "gpt-j-6B")

def has_model():
    if not os.path.isdir(model_dir):
        return False
    if any(f.endswith(".safetensors") for f in os.listdir(model_dir)):
        return True
    if os.path.isfile(os.path.join(model_dir, "pytorch_model.bin")):
        return True
    return os.path.isfile(os.path.join(model_dir, "pytorch_model.bin.index.json"))

if has_model():
    print("Model already exists, skipping download.")
else:
    print("Downloading ainize/gpt-j-6B-float16 (~11.2GB, 5-20 min depending on bandwidth)...")
    os.makedirs(model_dir, exist_ok=True)
    snapshot_download(
        repo_id="ainize/gpt-j-6B-float16",
        local_dir=model_dir,
        allow_patterns=["*.json", "*.txt", "*.safetensors", "*.bin", "*.md", "*.gitattributes"],
    )
    print("Model download complete.")
    for f in sorted(os.listdir(model_dir))[:20]:
        print(" -", f)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Model download complete.
 - .cache
 - .gitattributes
 - README.md
 - added_tokens.json
 - config.json
 - merges.txt
 - pytorch_model.bin
 - special_tokens_map.json
 - tokenizer.json
 - tokenizer_config.json
 - vocab.json


In [7]:
import os
import sys
# Ensure the data directory exists before the script downloads datasets
os.makedirs(os.path.join(PROJ_DIR, "EasyEdit", "data"), exist_ok=True)

# 3.0 Sync the reproduction script to the latest remote version.
# The Colab repo may be stale (git pull missing), so discard any local
# uncommitted patch first, then pull --v-steps/--ctx-len support.
import subprocess
subprocess.run(["git", "-C", PROJ_DIR, "checkout", "--", "EasyEdit/examples/w2_rome_repro.py"], check=False)
subprocess.run(["git", "-C", PROJ_DIR, "pull"], check=False)

# 3.1 Run ROME (full output captured to /content/w2_run.log for debugging)
# T4 (16GB) VRAM-lean defaults: GPT-J-6B fp16 weights ~11.2GB,
# so shrink mom2 samples, v grad steps and context lengths.
import os
import torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

DATA_SUBSET   = "zsre"     # counterfact / zsre / mquake
N_CASES       = 10         # 10 for smoke test, 50 for full
MOM2_SAMPLES  = 300        # mom2 statistics sample count (smaller = less VRAM/time)
V_STEPS       = 12         # v_num_grad_steps override (default 20; lower = less VRAM)
CTX_LEN       = "3,5"      # context template length override (smaller = less VRAM)
USE_FP16      = True       # keep True on T4

cmd = [
    sys.executable, "examples/w2_rome_repro.py",
    "--model", "gpt-j-6B",
    "--data", DATA_SUBSET,
    "--n", str(N_CASES),
    "--mom2-n-samples", str(MOM2_SAMPLES),
]
if USE_FP16:
    cmd.append("--fp16")
if V_STEPS:
    cmd.extend(["--v-steps", str(V_STEPS)])
if CTX_LEN:
    cmd.extend(["--ctx-len", CTX_LEN])

print("Executing command:", " ".join(cmd))
print("Note: the first run includes data download + mom2 statistics; a single edit may take 5-30 min, keep the tab open.")
log_path = "/content/w2_run.log"
with open(log_path, "w") as logf:
    result = subprocess.run(cmd, cwd=os.path.join(PROJ_DIR, "EasyEdit"),
                            stdout=logf, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
print("ROME run exit code:", result.returncode)
print("Full log saved:", log_path)
with open(log_path) as f:
    lines = f.readlines()
print("total log lines:", len(lines))
for ln in lines[-40:]:
    print(ln, end="")

Executing command: /usr/bin/python3 examples/w2_rome_repro.py --model gpt-j-6B --data zsre --n 10 --mom2-n-samples 300 --fp16 --v-steps 12 --ctx-len 3,5
Note: the first run includes data download + mom2 statistics; a single edit may take 5-30 min, keep the tab open.
ROME run exit code: 0
Full log saved: /content/w2_run.log
total log lines: 566
    "target_new": "Aki Kaurism\u00e4ki",
    "rewrite_acc": [
      1.0
    ],
    "rephrase_acc": [
      1.0
    ],
    "nq_acc": [
      1.0
    ]
  },
  {
    "case_id": 8,
    "prompt": "What was the name of the architect who worked on Lahti Town Hall?",
    "target_new": "Aki Kaurism\u00e4ki",
    "rewrite_acc": [
      1.0
    ],
    "rephrase_acc": [
      0.8333333333333334
    ],
    "nq_acc": [
      1.0
    ]
  },
  {
    "case_id": 9,
    "prompt": "Which designer was responsible for Lahti Town Hall?",
    "target_new": "Aimee Teegarden",
    "rewrite_acc": [
      1.0
    ],
    "rephrase_acc": [
      1.0
    ],
    "nq_acc": [
   

In [8]:
# 4.1 Copy the latest result to /content and print a metric summary (matches script fields)
import glob, json, shutil

res_dir = os.path.join(PROJ_DIR, "EasyEdit", "results", "eval")
jsons = sorted(glob.glob(os.path.join(res_dir, "ROME_gpt-j-6B_*.json")), key=os.path.getmtime)
if not jsons:
    raise FileNotFoundError("No result JSON found. Check the previous step (3.1) log.")

latest = jsons[-1]
dst = "/content/ROME_gpt-j-6B_latest.json"
shutil.copy(latest, dst)
print("Latest result file:", os.path.basename(latest))
print("Copied to:", dst)

with open(latest, "r", encoding="utf-8") as f:
    data = json.load(f)

print("\n===== Metric Summary =====")
print("model        :", data.get("model"))
print("split        :", data.get("split"))
print("rewrite_acc  :", data.get("rewrite_acc"))
print("rephrase_acc :", data.get("rephrase_acc"))
print("locality     :", data.get("locality"))
print("portability  :", data.get("portability"))
print("fp16         :", data.get("fp16"))
print("elapsed_s    :", data.get("elapsed_s"))
print("peak_vram_mb :", data.get("peak_vram_mb"))

Latest result file: ROME_gpt-j-6B_20261019_131404.json
Copied to: /content/ROME_gpt-j-6B_latest.json

===== Metric Summary =====
model        : gpt-j-6B
split        : zsre (official, first 10)
rewrite_acc  : 1.0
rephrase_acc : 0.95
locality     : {'nq_acc': 0.61}
portability  : {}
fp16         : True
elapsed_s    : 52.2
peak_vram_mb : 13098


In [9]:
# 4.2 Option 1: Save to Google Drive (optional - does NOT block Run All)
# Drive mounting pops an authorization dialog. To enable Drive saving, set
# SAVE_TO_DRIVE=true in this cell (or an earlier one), otherwise Run All skips it.
import os
if os.environ.get("SAVE_TO_DRIVE", "").lower() in ("1", "true", "yes"):
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    drive_dir = "/content/drive/MyDrive/CISC7021_ROME"
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copy(dst, os.path.join(drive_dir, os.path.basename(dst)))
    print("Saved result JSON to Drive:", drive_dir)
else:
    print("Drive save skipped (set SAVE_TO_DRIVE=true to enable).")


Drive save skipped (set SAVE_TO_DRIVE=true to enable).


In [10]:
# 4.3 Option 2: Download directly to the local browser
from google.colab import files
files.download(dst)
print("Download triggered; if nothing happens, manually download from the left file panel: " + dst)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered; if nothing happens, manually download from the left file panel: /content/ROME_gpt-j-6B_latest.json


## Notes
- **v6**: cleaned up for one-click Run All. Removed temporary/diagnostic cells
  (nvidia-smi probe, torch --force-reinstall, stray `pip install higher/torchvision`
  patch cells). `higher` is now part of the 1.4 dependency list (required by the
  MEND.py import chain). Drive save (4.2) is optional and skipped by default so
  Run All does not hang on authorization.
- **v5**: fixed gpu/hparams fields in the result JSON (script-side auto-detection,
  commit a79b7e4) and tightened dependency install.
- **Deps**: 1.4 installs requirements.txt (torch lines excluded) plus the BLIP2/
  MEND import-chain extras, then verifies `import easyeditor` and auto-fills any
  missing package (up to 5 rounds).
- **Model**: 2.2 downloads GPT-J-6B safetensors + pytorch_model.bin idempotently.
- **Run All**: if a later cell fails, read /content/w2_run.log (3.1) or the 4.1 error.
